# MetroPT-3 Data Audit

Reproducible audit record for the MetroPT-3 telemetry used by AeroXAI.

This notebook replaces the orchestration in `ml/data/inspect_metropt.py`.
Reusable normalization/schema logic remains in `ml/data/schema.py`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()

if not (ROOT / "ml").exists() and (ROOT.parent / "ml").exists():
    ROOT = ROOT.parent

if not (ROOT / "ml").exists():
    raise RuntimeError(
        "Could not locate repository root. "
        "Open this notebook from the xAI-Compressor repository."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Repository root: {ROOT}")

Repository root: /mnt/c/Users/Dell ProMax Tower T2/Downloads/code/xAI-Compressor


In [ ]:
import json

import pandas as pd
import yaml

from ml.data.schema import (
    ANALOG_COLUMNS,
    DIGITAL_COLUMNS,
    canonicalize_column_name,
    normalize_raw_frame,
)

CONFIG_PATH = ROOT / "configs" / "metropt.yaml"
REPORT_PATH = ROOT / "docs" / "data_audit.json"

with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)["dataset"]

csv_path = ROOT / config["raw_csv"]
print(f"Reading: {csv_path}")

Reading: /mnt/c/Users/Dell ProMax Tower T2/Downloads/code/xAI-Compressor/data/raw/MetroPT3(AirCompressor).csv


In [3]:
raw = pd.read_csv(csv_path)

canonical_raw = raw.rename(
    columns={
        column: canonicalize_column_name(column)
        for column in raw.columns
    }
)

raw_timestamps = pd.to_datetime(
    canonical_raw["timestamp"],
    errors="raise",
)

duplicate_timestamps = int(
    raw_timestamps.duplicated().sum()
)

frame = normalize_raw_frame(raw)

print(f"Raw rows:        {len(raw):,}")
print(f"Normalized rows: {len(frame):,}")
print(f"Duplicates:      {duplicate_timestamps:,}")
print(
    f"Time range:      {frame['timestamp'].min()} "
    f"→ {frame['timestamp'].max()}"
)

Raw rows:        1,516,948
Normalized rows: 1,516,948
Duplicates:      0
Time range:      2020-02-01 00:00:00 → 2020-09-01 03:59:50


## Timestamp cadence and gaps

In [4]:
timestamp_deltas = (
    frame["timestamp"]
    .diff()
    .dt.total_seconds()
    .dropna()
)

cadence_quantiles = (
    timestamp_deltas
    .quantile([0.01, 0.50, 0.95, 0.99])
    .to_dict()
)

cadence_top = {
    str(key): int(value)
    for key, value in (
        timestamp_deltas
        .value_counts()
        .head(10)
        .items()
    )
}

gaps = {
    "gt_15_seconds": int((timestamp_deltas > 15).sum()),
    "gt_30_seconds": int((timestamp_deltas > 30).sum()),
    "gt_60_seconds": int((timestamp_deltas > 60).sum()),
    "gt_300_seconds": int((timestamp_deltas > 300).sum()),
}

pd.DataFrame(
    {
        "metric": ["min", "p01", "median", "p95", "p99", "max"],
        "seconds": [
            float(timestamp_deltas.min()),
            float(cadence_quantiles[0.01]),
            float(cadence_quantiles[0.50]),
            float(cadence_quantiles[0.95]),
            float(cadence_quantiles[0.99]),
            float(timestamp_deltas.max()),
        ],
    }
)

,metric,seconds
0,min,8.0
1,p01,9.0
2,median,10.0
3,p95,10.0
4,p99,12.0
5,max,172918.0


In [5]:
pd.Series(gaps, name="count").to_frame()

,count
gt_15_seconds,363
gt_30_seconds,331
gt_60_seconds,331
gt_300_seconds,275


## Sensor summaries

In [6]:
analog_summary = {}

for column in ANALOG_COLUMNS:
    series = frame[column]
    analog_summary[column] = {
        "min": float(series.min()),
        "p01": float(series.quantile(0.01)),
        "median": float(series.median()),
        "mean": float(series.mean()),
        "p99": float(series.quantile(0.99)),
        "max": float(series.max()),
        "std": float(series.std()),
    }

pd.DataFrame(analog_summary).T

,min,p01,median,mean,p99,max,std
tp2,-0.032,-0.024,-0.012,1.367826,10.3520,10.676,3.250930
tp3,0.730,7.876,8.960,8.984611,10.1400,10.302,0.639095
h1,-0.036,-0.024,8.784,7.568155,10.1140,10.288,3.333200
dv_pressure,-0.032,-0.026,-0.020,0.055956,2.1120,9.844,0.382402
reservoirs,0.712,7.876,8.960,8.985233,10.1380,10.300,0.638307
oil_temperature,15.400,48.825,62.700,62.644182,76.1750,89.050,6.516261
motor_current,0.020,0.035,0.045,2.050171,6.1875,9.295,2.302053


In [7]:
digital_values = {}

for column in DIGITAL_COLUMNS:
    values = frame[column].dropna().unique()
    digital_values[column] = sorted(
        float(value) for value in values
    )[:50]

pd.Series(digital_values, name="unique_values").to_frame()

,unique_values
comp,"[0.0, 1.0]"
dv_electric,"[0.0, 1.0]"
towers,"[0.0, 1.0]"
mpg,"[0.0, 1.0]"
lps,"[0.0, 1.0]"
pressure_switch,"[0.0, 1.0]"
oil_level,"[0.0, 1.0]"
caudal_impulses,"[0.0, 1.0]"


## Missing values and incident coverage

In [8]:
missing_values = {
    column: int(count)
    for column, count in frame.isna().sum().items()
}

pd.Series(missing_values, name="missing_count").to_frame()

,missing_count
timestamp,0
tp2,0
tp3,0
h1,0
dv_pressure,0
reservoirs,0
oil_temperature,0
motor_current,0
comp,0
dv_electric,0


In [9]:
incidents = []

for incident in config["reported_incidents"]:
    start = pd.Timestamp(incident["start"])
    end = pd.Timestamp(incident["end"])

    mask = frame["timestamp"].between(
        start,
        end,
        inclusive="both",
    )

    incidents.append(
        {
            **incident,
            "rows_in_interval": int(mask.sum()),
        }
    )

pd.DataFrame(incidents)

,id,start,end,condition,rows_in_interval
0,1,2020-04-18 00:00:00,2020-04-18 23:59:00,air_leak_high_stress,8657
1,2,2020-05-29 23:30:00,2020-05-30 06:00:00,air_leak_high_stress,2360
2,3,2020-06-05 10:00:00,2020-06-07 14:30:00,air_leak_high_stress,17315
3,4,2020-07-15 14:30:00,2020-07-15 19:00:00,air_leak_high_stress,1622


## Write the audit record

In [10]:
report = {
    "dataset": config["name"],
    "rows_raw": len(raw),
    "rows_after_normalization": len(frame),
    "expected_rows": int(config["expected_rows"]),
    "raw_columns": list(raw.columns),
    "canonical_columns": list(frame.columns),
    "duplicate_timestamps_raw": duplicate_timestamps,
    "missing_values": missing_values,
    "time_start": str(frame["timestamp"].min()),
    "time_end": str(frame["timestamp"].max()),
    "cadence_seconds": {
        "min": float(timestamp_deltas.min()),
        "p01": float(cadence_quantiles[0.01]),
        "median": float(cadence_quantiles[0.50]),
        "p95": float(cadence_quantiles[0.95]),
        "p99": float(cadence_quantiles[0.99]),
        "max": float(timestamp_deltas.max()),
        "most_common": cadence_top,
    },
    "gaps": gaps,
    "analog_summary": analog_summary,
    "digital_unique_values": digital_values,
    "reported_incidents": incidents,
}

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

REPORT_PATH.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)


print(f"Audit written to: {REPORT_PATH}")

Audit written to: /mnt/c/Users/Dell ProMax Tower T2/Downloads/code/xAI-Compressor/docs/data_audit.json
